In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

# Début de la mesure du temps global d'exécution
start_time_global = time.time()

# =============================================================================
# PARTIE 1 : CHARGEMENT DES DONNÉES & PRÉPARATION
# =============================================================================
path = r"C:\Users\pc\projet\return_nasdaq.m"
path_cov = r"C:\Users\pc\projet\covariance_nasdaq.m" 

try:
    data_returns = pd.read_csv(path, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    benefice = data_returns.to_numpy()
    print("--- Loading Returns Successful ---")
    print(f"Matrix Dimensions: {benefice.shape}")
except Exception as e:
    print(f"Error loading returns: {e}")
    benefice = np.random.randn(2196, 1)

try:
    data_cov = pd.read_csv(path_cov, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    covariance = data_cov.to_numpy()
    print("--- Loading Covariance Successful ---")
    print(f"Matrix Dimensions: {covariance.shape}")
except Exception as e:
    print(f"Error loading covariance: {e}")
    covariance = np.diag(np.random.uniform(0.01, 0.05, 2196))

nb_asset = 2196
cardinal = 100

return_vect = benefice.flatten()
covar_matrix = covariance
variability = np.sqrt(np.diag(covar_matrix))

seuil = np.mean(return_vect)
target = (return_vect >= seuil).astype(int)

X = np.column_stack((return_vect, variability))
y = target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =============================================================================
# PARTIE 2 : ENTRAÎNEMENT DU MLP & PRÉDICTIONS CONTINUES
# =============================================================================
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("\n--- Entraînement du MLP ---")
model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

# Obtenir les probabilités brutes (scores de confiance) du MLP pour la mutation guidée
X_all_scaled = scaler.transform(X)
mlp_scores = model.predict(X_all_scaled, verbose=0).flatten()

# Sélection du Top 300
predictions_finales = (mlp_scores > 0.5).astype(int)
index_vector = [i for i in range(len(predictions_finales)) if predictions_finales[i] == 1]
return_index_vector = return_vect[index_vector]
indices_tries = sorted(range(len(return_index_vector)), key=lambda i: return_index_vector[i], reverse=True)
top_indices = indices_tries[:cardinal]
index_vector_vector = [index_vector[i] for i in top_indices]

vecteur_portefeuille = np.zeros(nb_asset)
for i in index_vector_vector:
    vecteur_portefeuille[i] = 1

# Normalisation des scores du MLP pour guider la direction de la mutation
scores_sub = mlp_scores[index_vector_vector]
mutation_guide_directions = (scores_sub - np.min(scores_sub)) / (np.max(scores_sub) - np.min(scores_sub) + 1e-8)

# =============================================================================
# PARTIE 3 : NSGA-II AVEC AFFICHAGE ALIGNÉ RENDEMENT ET RISQUE
# =============================================================================
print("--- Lancement de NSGA-II (Mutation Guidée par MLP) ---")

return_vect_filtered = return_vect.copy()
covar_matrix_filtered = covar_matrix.copy()
for i in range(nb_asset):
    if vecteur_portefeuille[i] == 0:
        return_vect_filtered[i] = 0
        covar_matrix_filtered[i, :] = 0
        covar_matrix_filtered[:, i] = 0

nb_sol = 150                  
number_iteration_max = 500     
prob_crossover = 0.95         
prob_mutation = 5 / cardinal  
mutation_strength = 0.25      

population = np.zeros((nb_sol, nb_asset))
for i in range(nb_sol):
    random_weights = np.random.uniform(0.1, 1.0, size=len(index_vector_vector))
    population[i, index_vector_vector] = random_weights / np.sum(random_weights)

def evaluate_population(pop):
    returns = np.dot(pop, return_vect_filtered)
    risks = np.zeros(len(pop))
    for idx in range(len(pop)):
        w = pop[idx, :].reshape(1, -1)
        risks[idx] = w @ covar_matrix_filtered @ w.T
    return returns, risks

def fast_non_dominated_sort(returns, risks):
    pop_size = len(returns)
    domination_features = [[] for _ in range(pop_size)]
    domination_counters = np.zeros(pop_size, dtype=int)
    fronts = [[]]
    for p in range(pop_size):
        for q in range(pop_size):
            if (returns[p] >= returns[q] and risks[p] <= risks[q]) and (returns[p] > returns[q] or risks[p] < risks[q]):
                domination_features[p].append(q)
            elif (returns[q] >= returns[p] and risks[q] <= risks[p]) and (returns[q] > returns[p] or risks[q] < risks[p]):
                domination_counters[p] += 1
        if domination_counters[p] == 0: fronts[0].append(p)
    i = 0
    while len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in domination_features[p]:
                domination_counters[q] -= 1
                if domination_counters[q] == 0: next_front.append(q)
        i += 1
        fronts.append(next_front)
    return fronts[:-1]

def calculate_crowding_distance(returns, risks, front):
    distance = np.zeros(len(front))
    if len(front) <= 2:
        distance[:] = np.inf
        return distance
    obj_returns = returns[front]
    sort_idx_ret = np.argsort(obj_returns)
    distance[sort_idx_ret[0]] = np.inf
    distance[sort_idx_ret[-1]] = np.inf
    ret_range = np.max(obj_returns) - np.min(obj_returns)
    if ret_range > 0:
        for i in range(1, len(front) - 1): distance[sort_idx_ret[i]] += (obj_returns[sort_idx_ret[i+1]] - obj_returns[sort_idx_ret[i-1]]) / ret_range
    obj_risks = risks[front]
    sort_idx_risk = np.argsort(obj_risks)
    distance[sort_idx_risk[0]] = np.inf
    distance[sort_idx_risk[-1]] = np.inf
    risk_range = np.max(obj_risks) - np.min(obj_risks)
    if risk_range > 0:
        for i in range(1, len(front) - 1): distance[sort_idx_risk[i]] += (obj_risks[sort_idx_risk[i+1]] - obj_risks[sort_idx_risk[i-1]]) / risk_range
    return distance

# Boucle principale de l'algorithme génétique
for generation in range(1, number_iteration_max + 1):
    returns, risks = evaluate_population(population)
    offspring = np.zeros_like(population)
    indices = np.arange(nb_sol)
    np.random.shuffle(indices)
    
    for k in range(0, nb_sol, 2):
        parent1 = population[indices[k]]
        parent2 = population[indices[k+1]]
        child1 = parent1.copy()
        child2 = parent2.copy()
        
        if np.random.rand() < prob_crossover:
            for asset_idx in index_vector_vector:
                pmin = min(parent1[asset_idx], parent2[asset_idx])
                pmax = max(parent1[asset_idx], parent2[asset_idx])
                diff = pmax - pmin
                child1[asset_idx] = np.random.uniform(pmin - 0.5 * diff, pmax + 0.5 * diff)
                child2[asset_idx] = np.random.uniform(pmin - 0.5 * diff, pmax + 0.5 * diff)
            
        for idx_sub, asset_idx in enumerate(index_vector_vector):
            direction = 1 if np.random.rand() < mutation_guide_directions[idx_sub] else -1
            if np.random.rand() < prob_mutation:
                child1[asset_idx] += direction * abs(np.random.normal(0, mutation_strength))
            if np.random.rand() < prob_mutation:
                child2[asset_idx] += direction * abs(np.random.normal(0, mutation_strength))
                
        child1[child1 < 1e-6] = 1e-6
        child2[child2 < 1e-6] = 1e-6
        child1[vecteur_portefeuille == 0] = 0
        child2[vecteur_portefeuille == 0] = 0
        if np.sum(child1) > 0: child1 /= np.sum(child1)
        if np.sum(child2) > 0: child2 /= np.sum(child2)
        offspring[k] = child1
        offspring[k+1] = child2

    combined_pop = np.concatenate((population, offspring), axis=0)
    comb_returns, comb_risks = evaluate_population(combined_pop)
    fronts = fast_non_dominated_sort(comb_returns, comb_risks)
    
    # --- AFFICHAGE REPRIS DE VOTRE VERSION PRÉFÉRÉE MAIS ALIGNÉ AVEC LES RISQUES ---
    first_front_indices = np.array(fronts[0])
    
    print(f"Iteration  {generation}")
    print(f"Non dominated selected solutions in iteration {generation} are: {first_front_indices}")
    print(f"Expected return of these solutions is: {comb_returns[first_front_indices]}")
    print(f"Risk of these solutions is:            {comb_risks[first_front_indices]}\n")
    
    new_population = []
    front_idx = 0
    while len(new_population) + len(fronts[front_idx]) <= nb_sol:
        new_population.extend(fronts[front_idx])
        front_idx += 1
        if front_idx >= len(fronts): break
    if len(new_population) < nb_sol and front_idx < len(fronts):
        last_front = fronts[front_idx]
        distances = calculate_crowding_distance(comb_returns, comb_risks, last_front)
        sorted_last_front = [last_front[i] for i in np.argsort(distances)[::-1]]
        new_population.extend(sorted_last_front[:(nb_sol - len(new_population))])
    population = combined_pop[new_population]

end_time_global = time.time()
print("="*60)
print(f"Temps d'exécution total option 1 : {end_time_global - start_time_global:.2f} secondes")
print("="*60)